# 03 - LightGBM 5-Fold CV

Same raw features and the same 5-fold `StratifiedKFold` split
(`shuffle=True, random_state=42`) as `02_kfold_cv.ipynb`'s CatBoost run,
on purpose: this notebook is a fair, apples-to-apples model-family
comparison, not a fresh experiment with its own split. Same fold
indices also matters for later stacking - see
`04-Ensembling-Calibration` in the vault.

In [ ]:
import pandas as pd
import numpy as np
import time
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

In [ ]:
train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

TARGET = "Will_Buy_EV"

X = train.drop(columns=[TARGET, "id"])
y = train[TARGET].map({"No": 0, "Yes": 1})
X_test = test.drop(columns=["id"])

# pandas 3 reads string columns as "str", not "object" - see 02_kfold_cv.ipynb.
cat_cols = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()

# LightGBM's native categorical support needs pandas "category" dtype
# specifically (unlike CatBoost, which took plain strings via cat_features).
# Test categories are aligned to the train categories so unseen levels
# don't silently become a different encoding.
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

print("Train:", X.shape, " Test:", X_test.shape)
print("Categorical columns:", cat_cols)

Train: (668665, 13)  Test: (286571, 13)
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 5-fold CV training

`num_leaves=63`, `learning_rate=0.05`, up to 3000 trees with
`early_stopping_rounds=50` per fold - LightGBM's leaf-wise growth tends
to need fewer, more targeted stopping rounds than CatBoost's
symmetric trees, hence the smaller `n_estimators` ceiling than the
CatBoost notebook's 2000 (LightGBM converged well under that anyway,
see best-iteration numbers below).

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_preds = np.zeros((N_SPLITS, len(X_test)))
fold_aucs = []
fold_best_iters = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=63,
        objective="binary",
        metric="auc",
        random_state=42,
        verbosity=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    va_pred = model.predict_proba(X_va)[:, 1]
    oof_pred[va_idx] = va_pred

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_aucs.append(fold_auc)
    fold_best_iters.append(model.best_iteration_)
    test_preds[fold] = model.predict_proba(X_test)[:, 1]

    print(f"Fold {fold}: AUC={fold_auc:.6f}  best_iter={model.best_iteration_}  elapsed={time.time()-t0:.0f}s")

print("Done in", round(time.time()-t0), "s")

Fold 0: AUC=0.940232  best_iter=334  elapsed=12s
Fold 1: AUC=0.941255  best_iter=241  elapsed=16s
Fold 2: AUC=0.942555  best_iter=406  elapsed=21s
Fold 3: AUC=0.941924  best_iter=193  elapsed=24s
Fold 4: AUC=0.941456  best_iter=245  elapsed=27s
Done in 27 s


In [ ]:
oof_auc = roc_auc_score(y, oof_pred)

print("Fold AUCs:", [round(a, 6) for a in fold_aucs])
print("Mean fold AUC:", np.mean(fold_aucs), " Std:", np.std(fold_aucs))
print("OOF AUC:", oof_auc)
print("Best iterations per fold:", fold_best_iters, " mean:", np.mean(fold_best_iters))

Fold AUCs: [0.940232, 0.941255, 0.942555, 0.941924, 0.941456]
Mean fold AUC: 0.9414844562592023  Std: 0.0007696495388404623
OOF AUC: 0.9414696433473926
Best iterations per fold: [334, 241, 406, 193, 245]  mean: 283.8


**LightGBM OOF AUC 0.941470 vs. CatBoost OOF AUC 0.941656 (Experiment
002)** - LightGBM comes in slightly *behind* CatBoost here, not ahead of
it. That's a legitimate result, not a bug: CatBoost's ordered target
statistics for categorical features tend to have an edge on datasets
like this one where several features are pure categoricals with no
natural ordinal structure (`City_Type`, `Current_Car_Type`, etc.).
LightGBM's real advantage shown here is speed - 27s vs. ~588s for the
same 5 folds - which matters a lot once hyperparameter tuning enters
the picture (Optuna sweeps are far cheaper with LightGBM).

In [ ]:
test_pred_avg = test_preds.mean(axis=0)

# How different are LightGBM's predictions from CatBoost's? High
# correlation here means an ensemble of the two won't add much - the
# two models are largely agreeing, not compensating for each other's
# mistakes.
catboost_sub = pd.read_csv("../submissions/catboost_5fold_baseline.csv")
corr = np.corrcoef(test_pred_avg, catboost_sub[TARGET].values)[0, 1]
print("Correlation with CatBoost test predictions:", corr)

Correlation with CatBoost test predictions: 0.9989745237581023


> [!warning] Limited ensemble diversity right now
> 0.9990 correlation is very high - on raw features, CatBoost and
> LightGBM are essentially learning the same function. Stacking these
> two as-is (Phase 4 of the execution plan) won't gain much. Diversity
> is more likely to come from: (a) different *feature sets* per model
> once Phase 2's engineered features exist, or (b) a genuinely
> different model family like the entity-embedding DNN reviewed in
> `05-Public-Notebooks-Review`, not from swapping between two
> gradient-boosted tree libraries on identical inputs.

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    TARGET: test_pred_avg
})

submission.to_csv("../submissions/lightgbm_5fold_baseline.csv", index=False)

print("Submission shape:", submission.shape)
print("Missing values:", submission.isna().sum().sum())
submission.head()

Submission shape: (286571, 2)
Missing values: 0


       id  Will_Buy_EV\n0  668665     0.007831\n1  668666     0.018444\n2  668667     0.004618\n3  668668     0.003230\n4  668669     0.015942

## Experiment 003 - LightGBM 5-Fold CV

- Features: identical raw features to Experiment 002, no engineering
- CV: identical 5-fold `StratifiedKFold` (same `random_state=42` -> same
  fold membership as Experiment 002, so the comparison below isn't
  confounded by different splits)
- Model: `LGBMClassifier`, `num_leaves=63`, `learning_rate=0.05`, up to
  3000 trees, `early_stopping_rounds=50` per fold, native pandas
  `category` dtype for categoricals

**Mean fold AUC: 0.941484 (std 0.000770) - OOF AUC: 0.941470**

| | Experiment 002 (CatBoost) | Experiment 003 (LightGBM) |
|---|---|---|
| OOF AUC | 0.941656 | 0.941470 |
| Training time (5 folds) | ~588s | ~27s |
| Mean best iteration | 907 | 284 |

**Verdict: not a better algorithm on these inputs** - CatBoost stays
ahead by ~0.0002 AUC, and the two models' test predictions correlate at
0.999, so there's not much ensembling upside from combining them as-is.
LightGBM's real payoff is speed: ~20x faster per fold, which makes it
the better choice for the Optuna hyperparameter sweeps in Phase 3, and
for quickly testing which engineered features from Phase 2 actually
help before spending CatBoost's longer training time confirming it.